# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1 — Fossil fuel dependency treemap
import plotly.graph_objects as go

fossil_colors = {'Coal': '#D55E00', 'Oil': '#0072B2', 'Natural Gas': '#009E73'}
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()
region_totals = df_fossil.groupby('Region', as_index=False)['TWh'].sum()
leader_region = region_totals.loc[region_totals['TWh'].idxmax(), 'Region']

fig1 = px.treemap(
    df_fossil, path=['Region', 'Country', 'Source'], values='TWh',
    color='Source', color_discrete_map=fossil_colors,
    title=f'{leader_region} Has the Largest Total Fossil Energy Use'
)
# Colour internal (Region/Country) nodes grey; keep fossil-source leaves coloured.
for trace in fig1.data:
    trace.marker.colors = [
        fossil_colors.get(label, '#B8B8B8') if not label.startswith('(') else '#B8B8B8'
        for label in trace.labels
    ]
    trace.texttemplate = '%{label}<br>%{value:,.1f} TWh'
    trace.hovertemplate = '%{label}<br>%{value:,.1f} TWh<extra></extra>'
fig1.update_layout(margin=dict(t=75, l=10, r=10, b=10), height=650)
fig1.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — Low-carbon treemap and horizontal ranking bar
low = df.loc[df['Source_Type'] == 'Low-carbon'].copy()
low_country = low.groupby('Country', as_index=False)['TWh'].sum()
low_country['All'] = 'Low-carbon'
leader = low_country.loc[low_country['TWh'].idxmax(), 'Country']

fig2a = px.treemap(
    low_country, path=['All', 'Country'], values='TWh',
    title='Low-carbon Energy Production by Country (Nuclear + Hydro)'
)
fig2a.update_traces(texttemplate='%{label}<br>%{value:,.1f} TWh',
                    hovertemplate='%{label}: %{value:,.1f} TWh<extra></extra>')
fig2a.update_layout(height=600, margin=dict(t=70, l=10, r=10, b=10))
fig2a.show()

ranked = low_country.sort_values('TWh', ascending=True)
fig2b = px.bar(
    ranked, x='TWh', y='Country', orientation='h',
    title=f'{leader} Leads Countries in Low-carbon Energy (Nuclear + Hydro)',
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': 'Country'},
    text='TWh'
)
fig2b.update_traces(marker_color='#0072B2', texttemplate='%{text:,.1f}',
                    textposition='outside', cliponaxis=False)
fig2b.update_layout(template='plotly_white', height=max(500, 28 * len(ranked) + 130),
                    margin=dict(l=130, r=90, t=80, b=60))
fig2b.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE* The horizontal bar chart shows the country ranking more clearly because the bars share a common baseline and are sorted by TWh. The treemap is better for comparing each country’s contribution to the overall low-carbon energy total, but small differences between rectangle areas are harder to judge.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3 — 100% stacked horizontal bar, coal share highest at top
fossil_region = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
fossil_region['Share'] = 100 * fossil_region['TWh'] / fossil_region.groupby('Region')['TWh'].transform('sum')
coal_shares = fossil_region.loc[fossil_region['Source'] == 'Coal', ['Region', 'Share']]
coal_shares = coal_shares.sort_values('Share', ascending=False)
region_order = coal_shares['Region'].tolist()
top_region = coal_shares.iloc[0]['Region']
top_share = coal_shares.iloc[0]['Share']

fig3 = px.bar(
    fossil_region, x='Share', y='Region', color='Source', orientation='h',
    category_orders={'Region': region_order, 'Source': ['Coal', 'Oil', 'Natural Gas']},
    color_discrete_map={'Coal': fossil_colors['Coal'], 'Oil': '#A7A7A7', 'Natural Gas': '#D5D5D5'},
    title=f'{top_region} Relies Most on Coal: {top_share:.1f}% of Fossil Energy',
    labels={'Share': 'Share of regional fossil energy (%)'}
)
fig3.update_traces(texttemplate=None)
for trace in fig3.data:
    if trace.name == 'Coal':
        trace.text = [f'{v:.1f}%' for v in trace.x]
        trace.textposition = 'inside'
    else:
        trace.text = None
fig3.update_yaxes(categoryorder='array', categoryarray=region_order[::-1])
fig3.update_xaxes(range=[0, 100], ticksuffix='%')
fig3.update_layout(barmode='stack', template='plotly_white', height=550,
                   legend_title_text='Fossil source', xaxis_title='Share of fossil energy')
fig3.show()


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE* The treemap answers which regions, countries, and fossil sources contribute the largest absolute amounts of fossil energy (TWh). The 100% stacked bar answers which region has the highest proportion of coal within its own fossil energy mix, regardless of its total energy use.